# 🎯 [OPS04_M09] AAR 기반 방공 요격 순위 판정 오류 분석 및 위협 가중치 매트릭스 재보정
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M09 |
| **미션 주제** | AAR 기반 방공 요격 순위 판정 오류 분석 및 위협 가중치 매트릭스 재보정 |
| **소요 시간** | 약 3.0시간 |
| **핵심 산출물** | `docs/ops04_aar_report.md, configs/ops04_tuned_params.yaml` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 방공 요격 순위 판정 오류 분석 및 위협 가중치 매트릭스 재보정
**소요시간**: 약 3.0시간 | **이번 산출물**: `docs/ops04_aar_report.md, configs/ops04_tuned_params.yaml`

### 📋 미션 개요
**사후검토(AAR)를 통해 순위 판정 오류의 원인을 유형별로 규명하고, 격자 탐색과 미학습 시험 검증을 거쳐 위협 가중치 매트릭스를 재보정합니다.**

- **배경 및 필요성:** 초기 휴리스틱 가중치는 비위협 과대 판정이나 근거리 기만체 미격리 등의 오판을 남길 수 있으므로, AAR 피드백을 정량적으로 역환류하여 모델 파라미터를 개선하고 잔여 한계를 운용 대책으로 보완해야 합니다.
- **핵심 과업:** 초기 가중치 오류를 '비위협 과대 판정·스푸핑 미격리·등급 경계 오차'로 분류한 뒤 학습용 상황에서 위협 유사도·스푸핑 감점·임계값을 격자 탐색으로 튜닝하고, 미학습 시험 상황 교차 검증 및 재발 사례 대책을 포함한 YAML 설정을 도출합니다.
- **학습 목표 (습득 역량):** AAR 기반 정량적 오류 분류 및 원인 분석 역량, 데이터 누수 없는 학습/시험 분리 기반 파라미터 격자 탐색 능력, 모델 한계를 운용 절차로 보완하는 리스크 관리 역량을 체득합니다.
- **검증 및 산출물:** AAR 적용 후 종합 정확도 10%p 이상 향상 및 재발 오류 점검 기준을 반영한 분석 보고서(`docs/ops04_aar_report.md`)와 재보정 파라미터 파일(`configs/ops04_tuned_params.yaml`)을 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 사후검토(AAR)를 통해 위협 우선순위 판정 오류의 원인을 유형별로 규명하고, 학습·시험 데이터 분리 원칙 아래 파라미터를 재보정하며 잔여 한계를 운용 대책으로 보완하는 지속 개선 역량을 체득합니다.

**[핵심 과업]** 초기 가중치 오류를 '비위협 과대 판정·스푸핑 미격리·등급 경계 오차'로 분류한 뒤 학습 상황에서 위협 유사도·스푸핑 감점·임계값을 격자 탐색하고, 미학습 시험 상황 정확도 검증 및 YAML 파라미터·운용 대책을 정리합니다.

- **AAR 기반 오류 원인 구조화 및 진단 역량:** 판정 실패 사례를 단순 오차율로만 보지 않고 비위협 과대 판정, 스푸핑 미격리, 등급 경계 오차 등 작전적 오류 유형으로 분해해 개선 대상을 특정하는 분석 능력을 익힙니다.
- **데이터 누수 방지 격자 탐색(Grid Search) 재보정 능력:** 학습용 상황과 미학습 시험 상황을 엄격히 분리해 과적합을 차단하면서 위협 가중치 매트릭스를 체계적으로 최적화하는 파라미터 튜닝 기술을 습득합니다.
- **잔여 오류 리스크 관리 및 YAML 형상 관리 역량:** 재보정 후에도 남는 경계 사례(근거리 기만체 등)를 운용자 확인 대책으로 보완하고, 조정된 파라미터를 코드와 분리된 YAML 설정으로 관리하는 실무 역량을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
AAR 사후검토를 통해 요격 순위 산정 지연 및 가치 평가 오류 사례를 분석하고 위협 가중치 매트릭스를 역환류 수정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR로 순위 판정 오류의 원인을 분류하고 위협 가중치 매트릭스를 재보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **도달시간만 반영한 초기 가중치로 가상 상황을 판정해 오류를 '비위협 과대 판정, 스푸핑 미격리, 등급 경계 오차'로 분류 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/ops04_aar_report.md 및 configs/ops04_tuned_params.yaml` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 9 코드를 실행해 초기 가중치의 오류 원인별 건수를 확인한다.
#### 이 단계의 역할
미션 9의 초기 가중치 판정(`cause()`)을 실행해 AAR 오류 원인별('비위협 과대 판정', '스푸핑 미격리', '등급 경계 오차') 건수를 확인합니다.

#### 해야 할 일

- 도달시간만 반영한 초기 가중치 모델의 정확도(75.6%)와 총 오류 건수(21건)의 세부 내역을 확인한다.
- AI를 Co-pilot으로 활용해 각 오류가 '비위협 과대 판정', '스푸핑 미격리', '등급 경계 오차' 중 어디에 해당하는지 분류하는 `cause()` 로직을 검토한다.
- AAR 오류 분류표와 개선 우선순위를 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "초기 위협 가중치 모델의 판정 오류를 '비위협 과대 판정', '스푸핑 미격리', '등급 경계 오차' 3가지 원인으로 분류하는 cause() 함수 로직을 검토하고, 각 오류 유형이 방공 작전에 미치는 위험도를 정리해줘."

  - **AI 결과 검증 포인트:** AI가 오류 원인을 분류할 때 특정 사례를 근거 없이 누락하거나 '원인 미상'으로 넘기지 않고 21건 전체가 3대 유형으로 정확히 분류되는지 합계를 대조합니다.

#### 핵심 확인 지표

> 초기 가중치 정확도 75.6% (오류 21건 원인 분류 완료), 시험 상황 정확도 84.9% -> 95.3% (+10.5%p)

#### 학습자가 제출해야 하는 내용

- 미션 9 초기 가중치 오류 원인별 집계 로그
- AI 프롬프트 질의 및 오류 분류 합계 검증 기록

#### Step 2. 재보정 전후 시험 상황 정확도를 비교한다.
#### 이 단계의 역할
학습용 상황에서 격자 탐색으로 재보정한 가중치를 미학습 시험 상황에 적용해 정확도 향상(+10%p 이상)을 검증합니다.

#### 해야 할 일

- 학습용 상황과 미학습 시험 상황이 코드 내에서 엄격히 분리되어 있는지 확인한다.
- AI에게 격자 탐색으로 도출된 위협 유사도 지수·스푸핑 감점·임계값 후보군 중 과적합 없이 시험 정확도가 가장 높은 조합을 비교 출력하는 코드를 요청한다.
- 재보정 전후 시험 상황 정확도 변화(84.9% -> 95.3%, +10.5%p)를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "학습용 상황에서 격자 탐색으로 찾은 가중치 후보들을 학습에 쓰지 않은 시험 상황에 평가해 재보정 전후 정확도 향상폭(%p)과 AAR 지적 오류 재발 건수를 비교 출력하는 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 격자 탐색 과정에서 시험 상황(Test set)을 참조하는 데이터 누수가 없는지, 그리고 향상폭이 목표 기준(+10%p 이상)을 충족하는지 확인합니다.

#### 핵심 확인 지표

> 시험 상황 정확도 84.9% -> 95.3% (+10.5%p, 목표 +10%p 이상 달성)

#### 학습자가 제출해야 하는 내용

- 재보정 전후 시험 상황 정확도 비교표 및 회귀 테스트 로그
- AI 코드의 학습/시험 분리 검증 기록

#### Step 3. 재발 사례와 운용 대책 문구를 확인하고 YAML 출력을 정리한다.
#### 이 단계의 역할
AAR 지적 오류의 재발 사례(근거리 기만체 1건)와 운용자 확인 대책을 확인하고 재보정 파라미터를 YAML 형식으로 정리합니다.

#### 해야 할 일

- `recurrence()` 점검 결과에서 재발한 사례(근거리 기만체 1건 -> 재발 0건 기준 미달)의 발생 원인을 확인한다.
- AI에게 근거리 기만체처럼 가중치 조정만으로 완전 제거가 어려운 경계 사례에 대한 운용자 확인 대책 문구와 `configs/ops04_tuned_params.yaml` 포맷 정리를 요청한다.
- AAR 분석 보고서(`docs/ops04_aar_report.md`)와 YAML 설정 파일의 수치 일치를 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "재보정 후에도 근거리 기만체 1건이 재발한 원인을 AAR 보고서용으로 요약하고, 운용자 교차 확인 대책과 최종 파라미터를 포함한 ops04_tuned_params.yaml 초안을 작성해줘."

  - **AI 결과 검증 포인트:** AI가 재발 1건을 임의로 '재발 0건 달성'으로 왜곡 보고하지 않고, 정직하게 '기준 미달(1건) 및 운용자 HITL 보완 대책 필요'로 기술했는지 확인합니다.

#### 핵심 확인 지표

> AAR 지적 오류 재발 1건(근거리 기만체) 확인 및 운용자 확인 대책·YAML 출력 정리

#### 학습자가 제출해야 하는 내용

- AAR 오류 분석 보고서 초안(`docs/ops04_aar_report.md`) 및 YAML 설정(`configs/ops04_tuned_params.yaml`)
- AI 생성 보고서 정직성 및 YAML 문법 검증 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
격자에 `w_cls = 1.5`를 추가하거나 허용 폭(`0.005`)을 조정해 재발 0건 달성 가능 여부와 시험 정확도 변화를 실험합니다.

#### 해야 할 일

- 격자 탐색 후보에 `w_cls = 1.5`를 추가하거나 근거리 허용 폭(`0.005`)을 넓혀 재보정을 재실행한다.
- AI에게 재발 0건 달성 시 시험 상황 정확도나 다른 유형의 오경보가 희생되지 않는지 트레이드오프 분석을 요청한다.
- 변경 전후 재발 건수와 시험 정확도 비교 메모를 작성해 최종 파라미터 채택 여부를 결정한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "격자 탐색에 w_cls=1.5를 추가하거나 허용 폭을 넓혔을 때 AAR 재발 건수가 0건이 되는지, 그리고 그때 미학습 시험 정확도와 오경보율에 부작용이 없는지 비교 실험 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 재발 0건을 맞추기 위해 특정 사례에 과적합되어 전체 시험 정확도가 크게 하락하지 않는지 확인하고, 알고리즘 튜닝과 HITL 운용 대책 간의 균형을 논리적으로 기술합니다.

#### 핵심 확인 지표

> w_cls 1.5·허용 폭 조정 전후 재발 건수(1건->0건 여부) 및 시험 정확도 비교 완료

#### 학습자가 제출해야 하는 내용

- 격자 확장 전후 재발 건수·정확도 비교 메모 및 최종 파라미터 선정 결과
- AI 질의 프롬프트 및 과적합 여부 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 오류 원인 규명의 객관성 및 개선 조치의 논리적 타당성

#### 미션 PASS 조건
- 정량 기준: AAR 피드백 적용 후 종합 정확도 10%p 이상 향상 및 잔여 재발 사례(근거리 기만체 등) 운용 보완 대책 수립(또는 튜닝으로 재발 0건)
- 진행 기준: AAR 보고서 작성 및 파라미터 튜닝 전후 회귀 테스트 통과
- 검증 방법: 회귀 테스트(Regression Test) 스위트 전체 자동 실행 및 비교
- AI 검증 기준: 학습/시험 분리 격자 탐색의 데이터 누수 차단과 잔여 재발 사례(T07061) 정직 보고·운용 대책 수립을 AI 검증으로 확인 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
# [TODO : 학생 실습 9] 핵심 함수의 TODO 영역을 AI Co-pilot과 함께 구현하고 검증하세요.
"""AAR(사후검토)로 초기 위협 가중치 매트릭스의 우선순위 판정 오류를 유형별로 분석하고, 학습용 가상 상황에서 가중치를 재보정한 뒤 별도 시험 상황에서 정확도 향상과 같은 오류의 재발 여부를 확인해 조정 파라미터를 YAML 문자열로 출력한다."""
import itertools
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """고도화: 자산 가치 x 도달시간 x 위협 유사도, 운동학 불일치(스푸핑 의심) 항적은 별도 표시한다."""
    w = w or {"tau": 300, "p_min": 0.5, "value_pow": 1.0, "jump_max": 2.0}
    _, tti = geometry(df)
    val = ASSETS["value"].to_numpy()[None] ** w["value_pow"]
    asset_score = val * np.exp(-tti / w["tau"])
    best = asset_score.argmax(axis=1)
    p = clf.predict_proba(features(df)[FEATS])[:, 1]
    spoof = (df["max_jump"] / df["spd"]).to_numpy() > w["jump_max"]
    out = df[["id"]].copy()
    out["asset"] = ASSETS["asset"].to_numpy()[best]
    out["tti_s"] = tti[np.arange(len(df)), best]
    out["p_threat"] = p
    out["spoof_flag"] = spoof
    out["score"] = np.where(spoof, 0, p * asset_score.max(axis=1))
    out["urgent_pred"] = (p >= w["p_min"]) & (out["tti_s"] <= TTI_URGENT) & ~spoof
    return out.sort_values("score", ascending=False)

LEVELS = ["긴급", "주의", "일반"]


def truth(df):
    return np.select([df["threat"] & (df["true_tti"] <= 300), df["threat"] & (df["true_tti"] <= 900)], ["긴급", "주의"], "일반")


def judge(df, clf, W):
    """[TODO : 학생 실습 9-1] exp(-TTI/300) * p^w_cls * (1 - w_spoof*spoof) 지수와 임계값(th_urgent, th_watch)으로 3단계 판정하세요."""
    pass  # <<< 여기에 judge() 로직을 작성한 뒤 pass를 제거하세요 >>>


def acc(df, clf, W):
    return (judge(df, clf, W) == truth(df)).mean()


clf = train_classifier()
W0 = {"w_cls": 0.0, "w_spoof": 0.0, "th_urgent": 0.37, "th_watch": 0.05}     # 초기: 도달시간만 반영
aar = make_raid(7, 80, spoof=6)                                              # AAR 대상 상황
pred0, y = judge(aar, clf, W0), truth(aar)
err = aar[pred0 != y].assign(pred=pred0[pred0 != y], true=y[pred0 != y])


def cause(r):
    """[TODO : 학생 실습 9-2] 오판 사례를 스푸핑 미격리 / 비위협 과대 판정 / 등급 경계 오차 3가지 원인으로 분류하세요."""
    pass  # <<< 여기에 cause() 로직을 작성한 뒤 pass를 제거하세요 >>>


err["원인"] = [cause(r) for r in err.itertuples()]
print(f"[AAR] 초기 가중치 정확도 {acc(aar, clf, W0):.1%}, 오류 {len(err)}건")
print(err["원인"].value_counts().to_string())
print(err[["id", "type", "true", "pred", "원인"]].head(5).to_string(index=False))

# 재보정: 학습용 가상 상황 3개(시험과 다른 시드)에서 격자 탐색
train = [make_raid(s, 80, spoof=6) for s in (201, 202, 203)]
grid = itertools.product([0.0, 0.5, 1.0, 2.0, 4.0], [1.0, 0.0], [0.30, 0.37, 0.45], [0.03, 0.05, 0.08])
cands = [{"w_cls": a, "w_spoof": b, "th_urgent": c, "th_watch": d} for a, b, c, d in grid]
aar_idx = [i for i in np.where(pred0 != y)[0] if cause(aar.iloc[i]) != "등급 경계 오차(도달시간 추정)"]


def recurrence(W):
    p = judge(aar, clf, W)
    return int(sum(p[i] != y[i] for i in aar_idx))


score = {i: np.mean([acc(r, clf, W) for r in train]) for i, W in enumerate(cands)}
top = max(score.values())
# 역환류: 학습 정확도 최고치와 0.5%p 이내인 후보 중 AAR 지적 오류 재발이 가장 적은 가중치를 채택
best = min((W for i, W in enumerate(cands) if score[i] >= top - 0.005), key=lambda W: (recurrence(W), -np.mean([acc(r, clf, W) for r in train])))
tests = [make_raid(s, 80, spoof=6) for s in (8, 9, 10)]
a0 = np.mean([acc(r, clf, W0) for r in tests]); a1 = np.mean([acc(r, clf, best) for r in tests])
pred1 = judge(aar, clf, best)
recur = recurrence(best)
boundary_left = int(sum((pred1[i] != y[i]) for i in np.where(pred0 != y)[0]))
print("\n[재보정 결과] (시험 상황 3개 평균, 학습에 쓰지 않은 상황)")
print(f"  종합 정확도 {a0:.1%} -> {a1:.1%} ({(a1 - a0) * 100:+.1f}%p, 목표 +10%p 이상) -> {'달성' if a1 - a0 >= 0.10 else '미달'}")
print(f"  AAR 대상 상황 정확도 {acc(aar, clf, W0):.1%} -> {acc(aar, clf, best):.1%}")
print(f"  AAR 지적 오류(기만·스푸핑·비위협) 재발 {recur}건 (목표 0건) -> {'달성' if recur == 0 else '미달'}, 등급 경계 오류 포함 잔여 {boundary_left}건")
for i in aar_idx:
    if pred1[i] != y[i]:
        r = aar.iloc[i]
        print(f"  재발 사례 {r.id}({r.type}): 자산 근접(도달 {geometry(aar.iloc[[i]])[1].min():.0f}s)으로 지수가 '주의' 임계를 넘음 -> 근거리 미확인 항적은 운용자 육안·추가 센서 확인으로 처리")
print(pd.crosstab(pd.Series(truth(tests[0]), name="true"), pd.Series(judge(tests[0], clf, best), name="pred")).to_string())
print("\n# configs/ops04_tuned_params.yaml")
print(f"""threat_matrix:
  w_cls: {best['w_cls']}        # 위협 유사도 반영 지수 (초기 0.0)
  w_spoof: {best['w_spoof']}      # 스푸핑 의심 감점 (초기 0.0)
  th_urgent: {best['th_urgent']}   # 긴급 임계 (exp(-TTI/300) 기준)
  th_watch: {best['th_watch']}    # 주의 임계
evaluation:
  baseline_acc: {a0:.3f}
  tuned_acc: {a1:.3f}
note: "[교육용가상자료] 추상 위협 지수 매개변수이며 실제 무기 제원과 무관"
""")

<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR] 초기 가중치 정확도 75.6%, 오류 21건
원인
비위협(기만체·조류·민항기) 과대 판정    13
스푸핑 항적 미격리                5
등급 경계 오차(도달시간 추정)         3
    id     type true pred                    원인
T07002 uav_type   주의   긴급     등급 경계 오차(도달시간 추정)
T07005 uav_type   주의   긴급     등급 경계 오차(도달시간 추정)
T07010    decoy   일반   주의 비위협(기만체·조류·민항기) 과대 판정
T07020    decoy   일반   주의 비위협(기만체·조류·민항기) 과대 판정
T07021    decoy   일반   주의 비위협(기만체·조류·민항기) 과대 판정

[재보정 결과] (시험 상황 3개 평균, 학습에 쓰지 않은 상황)
  종합 정확도 84.9% -> 95.3% (+10.5%p, 목표 +10%p 이상) -> 달성
  AAR 대상 상황 정확도 75.6% -> 95.3%
  AAR 지적 오류(기만·스푸핑·비위협) 재발 1건 (목표 0건) -> 미달, 등급 경계 오류 포함 잔여 3건
  재발 사례 T07061(decoy): 자산 근접(도달 42s)으로 지수가 '주의' 임계를 넘음 -> 근거리 미확인 항적은 운용자 육안·추가 센서 확인으로 처리
pred  긴급  일반  주의
true            
긴급    18   0   2
일반     0  46   1
주의     2   0  17

# configs/ops04_tuned_params.yaml
threat_matrix:
  w_cls: 1.0        # 위협 유사도 반영 지수 (초기 0.0)
  w_spoof: 1.0      # 스푸핑 의심 감점 (초기 0.0)
  th_urgent: 0.37   # 긴급 임계 (exp(-TTI/300) 기준)
  th_watch: 0.05    # 주의 임계
evaluation:
  baseline_acc: 0.849
  tuned_acc: 0.953
note: "[교육용가상자료] 추상 위협 지수 매개변수이며 실제 무기 제원과 무관"
```
</details>
